# LAB 1 — O Alto Comando decide: da manchete aos critérios

**Disciplina** Tomada de Decisão Baseada em Dados para Gestores de Segurança Pública (CAES-PMERJ / FGV)
**Aula** 1 — Da pressão por decisão aos critérios mensuráveis (3 de novembro de 2026)
**Duração do lab** 1h45, das 10:05 às 11:50, em grupo
**Entrega** Este notebook preenchido até a checagem final (Parte 6), antes da aula 2

> **Para quem é este lab:** gestores que decidem. Não é preciso programar. Você vai trocar `'TODO'` pelas respostas do grupo, sempre entre aspas simples, e apertar *Run* (ou Shift+Enter). Tudo o que o grupo decide fica gravado em tabelas no Databricks e volta nas próximas aulas. Onde houver `TODO`, o grupo preenche.

## Antes de começar

1. Importe este arquivo no Databricks Free Edition: **Workspace → (seu usuário) → Import → File → `lab-1.ipynb`**.
2. Conecte o notebook ao compute *Serverless* (canto superior direito) e rode as células da Parte 0, na ordem.
3. **Uma pessoa do grupo** compartilha a tela e digita; as demais discutem. As respostas ficam em tabelas no workspace de quem digitou; ao final, cada integrante importa uma cópia preenchida.

**O cenário.** A **Blue Blood** é uma força policial fictícia cuja área coincide com o estado do Rio de Janeiro: Comando de Área = RISP, batalhão = AISP, companhia = CISP. Organização, pessoas e números internos são fictícios; os dados de criminalidade são reais e públicos (ISP-RJ).

**Como as células funcionam.** As células que começam com `%sql` são SQL, a linguagem padrão deste curso. As marcadas **🟢 PRONTA** só se executam. Nas demais, o grupo troca `'TODO'` pela resposta, sempre entre aspas simples, e executa. Cada resposta vira uma tabela da zona `gold`, com nome iniciado por `lab1_`.

**Onde este lab está no arco da disciplina.**

| Etapa | Neste lab |
| --- | --- |
| 1. Pressão por decisão | Rodada 1 e decisão do grupo (Partes 1 e 3) |
| 2. Critérios de decisão | Critérios, pesos AHP e limiares (Parte 4) |
| 3. Coleta e armazenamento | O espaço `blue-blood` e a primeira série do ISP na Bronze (Parte 0); o dado mais crítico (Parte 2) |
| 4. Processamento e visualização | Aula 2 |
| 5. Decisão com critérios e risco | Ensaio com a pontuação de alternativas (Parte 5) |

## Objetivos de aprendizagem

Ao final deste lab, o grupo será capaz de:

1. **Criar** o espaço persistente da Blue Blood no Databricks (catálogo, zonas Bronze, Silver e Gold, e volume) e carregar nele a primeira série do ISP.
2. **Nomear** a decisão que uma pressão externa está pedindo: o quê, quem decide, prazo e custo de errar.
3. **Identificar** o dado mais crítico para essa decisão, quem o produz e quem pode acessá-lo.
4. **Definir** 2 a 3 critérios mensuráveis, cada um com indicador e limiar, **antes** de olhar os dados.
5. **Calcular** os pesos dos critérios com AHP e verificar se os julgamentos são coerentes.
6. **Usar** os pesos para pontuar alternativas e explicar a troca que o comando está fazendo.

## Parte 0 — Primeiro acesso: o espaço `blue-blood` e o Alto Comando (10:05–10:20)

### Atividade 0.1: criar o espaço da Blue Blood (5 min, células prontas)

Antes de decidir, o comando precisa de um lugar para guardar os dados e as decisões. No Databricks esse lugar é um **catálogo**. O nosso se chama `blue-blood` e é organizado nas zonas da arquitetura medalhão (Bronze, Silver e Gold), com um **volume** para os arquivos:

| Onde | O que guarda | Exemplo |
| --- | --- | --- |
| Catálogo `blue-blood` | Tudo da Blue Blood, persistente entre aulas | Aparece em **Catalog** no menu à esquerda |
| Zona `bronze` | O dado como chegou, sem alteração | `bronze.isp_cisp_mensal` |
| Zona `silver` | Limpo, tipado e checado | `silver.isp_ocorrencias` |
| Zona `gold` | Indicadores e **todas as decisões e parâmetros do grupo** | `gold.lab1_criterios`, `gold.lab1_pesos` |
| Volume `bronze.arquivos` | Os arquivos baixados (CSV, planilhas) | `/Volumes/blue-blood/bronze/arquivos/` |

Nada fica solto no notebook: o que o grupo decide hoje continua lá na aula 2, na 3 e na 4.

> Se `CREATE CATALOG` der erro de permissão ou de *storage root*, crie o catálogo pela tela: **Catalog → + → Create catalog**, nome `blue-blood`, tipo *Standard*, armazenamento *Default Storage*. Depois rode a célula de novo.

In [0]:
%sql
-- 🟢 PRONTA — cria (ou reabre) o espaço persistente da Blue Blood. Só execute.
-- Catálogo = onde ficam as tabelas. Zonas (schemas) = Bronze, Silver e Gold. Volume = onde ficam os arquivos.
-- Se reconectar o notebook, rode esta célula de novo: ela não apaga nada.
CREATE CATALOG IF NOT EXISTS `blue-blood` COMMENT 'Blue Blood (CAES-PMERJ / FGV): tabelas e arquivos dos labs';
USE CATALOG `blue-blood`;
CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Bronze: o dado como chegou, sem alteração';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Silver: limpo, tipado e checado';
CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Gold: indicadores e as decisões e parâmetros do grupo';
CREATE VOLUME IF NOT EXISTS bronze.arquivos COMMENT 'Arquivos baixados como chegaram (ISPDados, narrativas)';
SHOW SCHEMAS

### Atividade 0.2: carregar um exemplo inicial de dados do ISP (5 min, células prontas)

As três células abaixo fazem o caminho completo de um dado público: **baixar o arquivo para o volume** (Bronze), **transformar em tabela** sem mexer em nada (Bronze) e **padronizar** em uma linha por CISP, mês e delito (Silver). A fonte é a série mensal por delegacia do [ISPDados](https://www.ispdados.rj.gov.br/), livre e sem cadastro.

> Hoje o grupo **não olha os números** do seu recorte: os critérios vêm antes dos dados (Parte 4). A carga serve para conhecer o caminho e para listar as CISPs e os delitos disponíveis na Parte 3.

In [0]:
# 🟢 PRONTA — baixa as séries públicas do ISPDados para o volume da Bronze. Só execute.
# O Databricks Free Edition limita o acesso à internet: se o site do ISP estiver bloqueado ou fora do ar,
# a célula grava no volume o PLANO B, uma amostra real do ISPDados embutida aqui (CISPs da RISP 1 e as
# CISPs 33 e 64, jan/2024 a set/2026, 14 delitos), baixada em 26/10/2026. Python só para o download; o resto é SQL.
import base64, gzip, urllib.request

VOLUME = "/Volumes/blue-blood/bronze/arquivos"
URLS = {
    "BaseDPEvolucaoMensalCisp.csv": "https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv",
    "PopulacaoEvolucaoMensalCisp.csv": "https://www.ispdados.rj.gov.br/Arquivos/PopulacaoEvolucaoMensalCisp.csv",
}
PLANO_B = {  # gzip + base64 dos dois arquivos, no mesmo formato do ISP (separador ;, latin-1)
    "BaseDPEvolucaoMensalCisp.csv": "...",
    "PopulacaoEvolucaoMensalCisp.csv": "H4sIAAAAAAACA03US64kRwgF0HmshUHwB9ViLOvJAw/sbtn7l3x5cpI9LApFAnGIrz//+fr89ce/n9///vH5+ePnb18IHP74R67Ih9ONnLOMjz1B1RIndb18/AnatQqkmtuJJ1gRl6mzLU5uplY7cXf76f2QhSIa0nb4bpRv4lPFVYe3qMQJSXUnVZ5gl2aT3BI+rG+q3yBJkTy8DbSLFLmmHt4GypKT3KQQzbcD1ku3Oftw7bESylSmF7m/9NDmQVUdR+4vI2xEucWObA9S7mFkrHlkmxBPDqH5mLw9pPqliEoca+9oUL5Sa/CRbYK1NZmsw4/EG42MJrs3j+65Ejd0Couj2y8G1R2kdv1ovVFtUaoWP/aWMHO95Cl6YqOWlSrkLQFHsUNQynu9x1Gso9tIxDThKNZRJIUHRhtPsPCbKfBfw1GsozTCwVFwFK8jSZr/xlGso0CumA6jWEYoivH9GEexjjBWLTUbR5sLHeQ4xMfRk+tcRlyXeSA9xaKBolsc35C2hVtN06sMpOdc8UYuLlMG0jbRIxzj7YH0znAaFsUlyHYBSAaL2hUDKRbSNUFUYyTFSmIjLBpmK/YOBxO81HM3sl2MJKyO56iLNxp5saatPZJiJWHxKYe4bsegVH2JcWtDaaNajFsvbK+9NQjgowvWb0qxlBxC4s6afXLHINgx1e8nKR9K2fBp7JCUK8kKRUEIJOVKEnznMoK5mYohIpg4svc7FrgcmKmBlC8kESxjykjKlcR4T/iyjKR8JAku0b0iR9LmGpauQkbH1g9J2lQi/S0pV5IW3sRr35K2BawBlSseVUjKlYT3BC9HDKR8IfH3NulAeieIqxpeMZByIQksjvyB9EQ9ehoG54H0fCxxtwj2ALV3OBrYPfQtAykXkhiWF3gH0kYjEO1KHki5kMAONeC9120YkLLw1P0PaaN46Zyk0Ju9NQDSFbpYioGUCwl5KBgP63+EPbVX8gYAAA==",
}

try:
    conteudos = {}
    for nome, url in URLS.items():
        with urllib.request.urlopen(url, timeout=60) as r:
            conteudos[nome] = r.read()
    ORIGEM_ISP = "ISPDados (download agora)"
except Exception as erro:
    conteudos = {nome: gzip.decompress(base64.b64decode(b)) for nome, b in PLANO_B.items()}
    ORIGEM_ISP = f"Plano B embutido (ISPDados de 26/10/2026); download falhou: {type(erro).__name__}"

spark.sql("""CREATE TABLE IF NOT EXISTS bronze.registro_ingestao
             (arquivo STRING, origem STRING, url STRING, bytes BIGINT, baixado_em TIMESTAMP)
             COMMENT 'Bronze: de onde veio cada arquivo e quando'""")
for nome, dados in conteudos.items():
    with open(f"{VOLUME}/{nome}", "wb") as f:
        f.write(dados)
    spark.sql("INSERT INTO bronze.registro_ingestao VALUES (:arquivo, :origem, :url, :bytes, current_timestamp())",
              args={"arquivo": nome, "origem": ORIGEM_ISP, "url": URLS[nome], "bytes": len(dados)})
print(ORIGEM_ISP)
print(f"Arquivos gravados em {VOLUME}: {', '.join(conteudos)}")

In [0]:
%sql
-- 🟢 PRONTA — Bronze: cada arquivo do volume vira uma tabela, sem alterar nenhum valor (tudo como texto). Só execute.
CREATE OR REPLACE TABLE bronze.isp_cisp_mensal
COMMENT 'Bronze: ISPDados BaseDPEvolucaoMensalCisp.csv como chegou (ocorrências por CISP e mês)'
AS SELECT *, current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/blue-blood/bronze/arquivos/BaseDPEvolucaoMensalCisp.csv',
                format => 'csv', header => true, sep => ';', encoding => 'ISO-8859-1', inferColumnTypes => false);

CREATE OR REPLACE TABLE bronze.isp_populacao_cisp
COMMENT 'Bronze: ISPDados PopulacaoEvolucaoMensalCisp.csv como chegou (população estimada por CISP)'
AS SELECT *, current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/blue-blood/bronze/arquivos/PopulacaoEvolucaoMensalCisp.csv',
                format => 'csv', header => true, sep => ';', inferColumnTypes => false);

SELECT 'bronze.isp_cisp_mensal' AS tabela, count(*) AS linhas FROM bronze.isp_cisp_mensal
UNION ALL
SELECT 'bronze.isp_populacao_cisp', count(*) FROM bronze.isp_populacao_cisp

In [0]:
%sql
-- 🟢 PRONTA — Silver: uma linha por CISP, mês e delito, com tipos certos. Só execute.
-- fase 2 = mês preliminar do ISP (ainda pode mudar); fase 3 = consolidado.
CREATE OR REPLACE TABLE silver.isp_ocorrencias
COMMENT 'Silver: ocorrências por CISP, mês e delito (ISPDados), tipadas; fase 2 = preliminar'
AS
SELECT cisp, aisp, risp, munic, make_date(ano, mes, 1) AS mes_ref, ano, mes, fase, delito, ocorrencias
FROM (
  SELECT CAST(cisp AS INT) AS cisp, CAST(aisp AS INT) AS aisp, CAST(risp AS INT) AS risp, munic,
         CAST(ano AS INT) AS ano, CAST(mes AS INT) AS mes, CAST(fase AS INT) AS fase,
         CAST(roubo_rua AS INT) AS roubo_rua,
         CAST(roubo_transeunte AS INT) AS roubo_transeunte,
         CAST(roubo_celular AS INT) AS roubo_celular,
         CAST(roubo_em_coletivo AS INT) AS roubo_em_coletivo,
         CAST(roubo_veiculo AS INT) AS roubo_veiculo,
         CAST(roubo_comercio AS INT) AS roubo_comercio,
         CAST(total_roubos AS INT) AS total_roubos,
         CAST(furto_transeunte AS INT) AS furto_transeunte,
         CAST(furto_celular AS INT) AS furto_celular,
         CAST(total_furtos AS INT) AS total_furtos,
         CAST(letalidade_violenta AS INT) AS letalidade_violenta,
         CAST(hom_doloso AS INT) AS hom_doloso,
         CAST(lesao_corp_dolosa AS INT) AS lesao_corp_dolosa,
         CAST(estupro AS INT) AS estupro
  FROM bronze.isp_cisp_mensal
)
UNPIVOT (ocorrencias FOR delito IN (roubo_rua, roubo_transeunte, roubo_celular, roubo_em_coletivo, roubo_veiculo, roubo_comercio, total_roubos, furto_transeunte, furto_celular, total_furtos, letalidade_violenta, hom_doloso, lesao_corp_dolosa, estupro))
WHERE cisp IS NOT NULL AND mes BETWEEN 1 AND 12;

CREATE OR REPLACE TABLE silver.isp_populacao
COMMENT 'Silver: última população estimada pelo ISP para cada CISP (a série vai até jul/2022)'
AS
SELECT CAST(circ AS INT) AS cisp,
       max_by(CAST(replace(pop_circ, ',', '.') AS DOUBLE), CAST(ano AS INT) * 100 + CAST(mes AS INT)) AS populacao,
       max(make_date(CAST(ano AS INT), CAST(mes AS INT), 1)) AS referencia
FROM bronze.isp_populacao_cisp
WHERE trim(pop_circ) <> ''
GROUP BY CAST(circ AS INT);

SELECT count(DISTINCT cisp) AS cisps, count(DISTINCT delito) AS delitos,
       min(mes_ref) AS primeiro_mes, max(mes_ref) AS ultimo_mes,
       count_if(fase = 2) AS linhas_preliminares
FROM silver.isp_ocorrencias

In [0]:
%sql
-- 🟢 PRONTA — Gold: o cardápio de recortes (CISPs e delitos disponíveis), sem nenhuma contagem. Só execute.
CREATE OR REPLACE TABLE gold.recortes_disponiveis
COMMENT 'Gold: CISPs com série no ISP e sua AISP, RISP e município, para o grupo escolher o recorte'
AS
SELECT cisp, max_by(aisp, mes_ref) AS aisp, max_by(risp, mes_ref) AS risp, max_by(munic, mes_ref) AS municipio,
       min(mes_ref) AS primeiro_mes, max(mes_ref) AS ultimo_mes
FROM silver.isp_ocorrencias
GROUP BY cisp;

CREATE OR REPLACE TABLE gold.delitos_disponiveis
COMMENT 'Gold: delitos (nomes de coluna do ISP) disponíveis na Silver'
AS SELECT DISTINCT delito FROM silver.isp_ocorrencias;

SELECT * FROM gold.recortes_disponiveis ORDER BY cisp

### Atividade 0.3: formação do Alto Comando (10 min)

Cada grupo de 5 é um Alto Comando da Blue Blood. Um integrante por papel; o grupo pode trocar o nome do personagem, não o papel.

| Papel | O que faz no grupo |
| --- | --- |
| Comandante-Geral | Fecha a decisão e apresenta na aula 4 |
| Chefe de Inteligência e Análise Criminal | Conduz o notebook e a checagem no ISP |
| Chefe de Comunicação Social | Traz as notícias e escreve a nota pública |
| Corregedora e Governança de Dados | Garante o Termo, a anonimização e o registro |
| Comandante de Área Operacional | Diz o custo operacional de cada decisão |

In [0]:
%sql
-- Atividade 0.3 — o número do grupo e quem assume cada papel.
-- Troque 0 pelo número do grupo e cada 'TODO' por um nome, ENTRE ASPAS SIMPLES. Depois Shift+Enter.
CREATE OR REPLACE TABLE gold.lab1_grupo COMMENT 'Lab 1: número do grupo' AS
SELECT 0 AS grupo;   -- TODO: número do grupo, de 1 a 8 (sem aspas)

CREATE OR REPLACE TABLE gold.lab1_alto_comando COMMENT 'Lab 1: quem assume cada papel do Alto Comando' AS
SELECT * FROM VALUES
  ('Comandante-Geral',                         'TODO'),
  ('Chefe de Inteligência e Análise Criminal', 'TODO'),
  ('Chefe de Comunicação Social',              'TODO'),
  ('Corregedora e Governança de Dados',        'TODO'),
  ('Comandante de Área Operacional',           'TODO')
AS t(papel, nome);

SELECT (SELECT grupo FROM gold.lab1_grupo) AS grupo, papel, nome FROM gold.lab1_alto_comando

## Parte 1 — Rodada 1 da Operação Manchete (10:20–10:40)

**Segunda-feira, 07:40.** Um portal publica: *"Roubos de rua disparam 40% em [bairro] nos últimos três meses"*. Em duas horas, milhares de compartilhamentos. O conselho comunitário pede reforço, a promotoria pergunta pelo planejamento e a repórter quer resposta até o meio-dia. Ninguém sabe de onde saiu o número.

> A manchete e os 40% são fictícios. Cada porta vale de 0 a 3 pontos; o professor revela depois que todos os grupos escolherem.

| Porta | Escolha |
| --- | --- |
| A | Deslocar efetivo de outra área para o bairro imediatamente |
| B | Pedir à Inteligência a série do ISP do bairro e prometer resposta à imprensa em 24 h |
| C | Desmentir a notícia em nota pública, sem números |
| D | Não comentar até sair a estatística oficial do mês |
| E | Encomendar um painel novo à TI antes de qualquer análise |

**Atividade 1.1 (10 min).** Discutam, escolham **uma** porta e justifiquem em **uma frase**. Cada papel deve dizer o que a decisão custa na sua área.

In [0]:
%sql
-- Atividade 1.1 — a escolha do grupo, ANTES de o professor revelar as consequências.
CREATE OR REPLACE TABLE gold.lab1_rodada1 COMMENT 'Lab 1: porta escolhida na Rodada 1 da Operação Manchete' AS
SELECT
  'TODO' AS porta_escolhida,   -- TODO: 'A', 'B', 'C', 'D' ou 'E'
  'TODO' AS justificativa      -- TODO: uma frase

**Atividade 1.2 (10 min).** O professor revela as consequências de cada porta. Digitem os pontos anunciados para a porta do grupo (0 a 3).

**Carta de papel.** Uma vez no jogo inteiro, o grupo pode somar 1 ponto a uma porta que valeu 0 ou 1, se explicar como um papel específico do Alto Comando evitaria o dano.

In [0]:
%sql
-- Atividade 1.2 — pontos anunciados pelo professor para a porta escolhida.
-- Troque só a palavra NULL. Ex.: CAST(2 AS INT), CAST('Corregedora e Governança de Dados' AS STRING).
CREATE OR REPLACE TABLE gold.lab1_rodada1_pontos COMMENT 'Lab 1: pontos da Rodada 1 e uso da carta de papel' AS
SELECT
  CAST(NULL AS INT)    AS pontos,            -- TODO: número de 0 a 3
  CAST(NULL AS STRING) AS carta_de_papel,    -- opcional: papel que evitaria o dano
  CAST(NULL AS STRING) AS carta_explicacao;  -- opcional: como esse papel evitaria o dano (uma frase)

SELECT r.porta_escolhida AS porta, p.pontos,
       CASE WHEN p.pontos IS NULL THEN 'Aguardando os pontos anunciados pelo professor.'
            WHEN p.carta_de_papel IS NOT NULL AND p.carta_explicacao IS NOT NULL AND p.pontos <= 1
              THEN concat('Carta de papel usada (', p.carta_de_papel, '): +1 ponto. Total: ', p.pontos + 1,
                          ' (máximo 3). Ela não pode ser usada de novo.')
            ELSE concat('Total na Rodada 1: ', p.pontos, ' ponto(s) (máximo 3).') END AS resultado
FROM gold.lab1_rodada1 r CROSS JOIN gold.lab1_rodada1_pontos p

**Para a plenária:** a porta que o grupo escolheu foi a mesma da enquete individual do início da aula? O que mudou quando cada papel falou do custo na sua área?

---

## Parte 2 — Qual é o dado mais crítico no seu dia a dia? (10:40–11:00)

**Atividade 2.1 (10 min).** Cada integrante cita **um** dado que usa (ou deveria usar) para decidir no trabalho real. Preencham uma linha por pessoa.

| Integrante | Dado | Quem produz | Onde fica | Com que atraso chega | Quem pode acessar | Que decisão sustenta |
| --- | --- | --- | --- | --- | --- | --- |
| | | | | | | |
| | | | | | | |
| | | | | | | |
| | | | | | | |
| | | | | | | |

> Dê dois cliques nesta célula para editar a tabela e depois Shift+Enter. Esta tabela é só para a conversa; o que vale como registro é a célula da Atividade 2.2.

**Atividade 2.2 (10 min).** O grupo escolhe o dado **mais crítico** e responde na célula abaixo.

> Lembre do slide de governança: se você sabe que o dado existe, deveria ter acesso e mesmo assim não consegue a resposta, você tem um problema de governança de dados.

In [0]:
%sql
-- Atividade 2.2 — o dado mais crítico do grupo.
CREATE OR REPLACE TABLE gold.lab1_dado_mais_critico COMMENT 'Lab 1: o dado mais crítico para decidir, quem produz e por que não chega' AS
SELECT
  'TODO' AS dado,
  'TODO' AS quem_produz,
  'TODO' AS decisao_que_sustenta,
  'TODO' AS esta_no_isp,          -- 'sim', 'não' ou 'em parte'
  'TODO' AS por_que_nao_chega     -- se o dado existe mas não chega a quem decide: por quê?

---

## Parte 3 — A decisão do grupo (11:00–11:10)

Agora o grupo escolhe o recorte que vai acompanhar até a aula 4: **o bairro ou a CISP** da manchete e **o tipo de roubo**. As CISPs e os delitos disponíveis estão nas tabelas `gold.recortes_disponiveis` e `gold.delitos_disponiveis`, criadas na Parte 0. Para rever, abra uma célula nova e rode `%sql SELECT * FROM gold.delitos_disponiveis`.

Antes de buscar qualquer dado, nomeiem a decisão.

In [0]:
%sql
-- Parte 3 — nomeie a decisão antes de olhar os dados.
CREATE OR REPLACE TABLE gold.lab1_decisao COMMENT 'Lab 1: a decisão em jogo, nomeada antes de olhar os dados' AS
SELECT
  'TODO' AS bairro_ou_cisp,      -- ex.: 'CISP 12 — Copacabana'
  'TODO' AS delito,              -- ex.: 'roubo_rua' (nome usado pelo ISP)
  'TODO' AS decisao_em_jogo,     -- o que muda se o comando agir?
  'TODO' AS quem_decide,
  'TODO' AS prazo,
  'TODO' AS custo_de_errar       -- o que acontece se a decisão estiver errada?

---

## Parte 4 — Critérios, pesos e limiares (11:10–11:40)

### Atividade 4.1: escolham 2 a 3 critérios (5 min)

Um critério diz **o que é uma boa decisão**. O exemplo abaixo é o da aula; o grupo pode manter, trocar ou acrescentar (no máximo 5).

| Critério do exemplo | Pergunta | Quem cobra no Alto Comando |
| --- | --- | --- |
| Evidência | O dado sustenta a decisão? | Chefe de Inteligência |
| Custo operacional | De onde sai o efetivo e o que fica descoberto? | Comandante de Área |
| Legitimidade | A população, a imprensa e a promotoria aceitam a decisão? | Chefe de Comunicação Social |

### Atividade 4.2: comparem os critérios dois a dois (10 min)

Para cada par, respondam: **"para esta decisão, quanto o primeiro importa mais que o segundo?"**, na escala de Saaty.

| Valor | Significado |
| --- | --- |
| 1 | Igual importância |
| 3 | Importância moderada |
| 5 | Importância forte |
| 7 | Importância muito forte |
| 9 | Importância extrema |
| 2, 4, 6, 8 | Intermediários |
| 1/3, 1/5, ... | O segundo importa mais que o primeiro |

Com 3 critérios são 3 julgamentos; com 4, são 6; com 5, são 10. A célula seguinte já vem com o exemplo da aula: troquem pelos critérios e julgamentos do grupo. Para dizer que o segundo importa mais, escrevam a fração, por exemplo `1/3`.

In [0]:
%sql
-- Atividade 4.2 — critérios e julgamentos par a par (exemplo da aula; troque pelos do grupo).
-- Os critérios do grupo (2 a 5). Troque os nomes, se quiser, e use os MESMOS nomes nos julgamentos.
CREATE OR REPLACE TABLE gold.lab1_criterios COMMENT 'Critérios de decisão do grupo' AS
SELECT * FROM VALUES
  ('Evidência'),
  ('Custo operacional'),
  ('Legitimidade')
AS t(criterio);

-- Julgamentos par a par na escala de Saaty: quanto o primeiro importa mais que o segundo.
CREATE OR REPLACE TABLE gold.lab1_julgamentos COMMENT 'Julgamentos par a par (escala de Saaty)' AS
SELECT criterio_a, criterio_b, CAST(valor AS DOUBLE) AS valor FROM VALUES
  ('Evidência', 'Custo operacional', 3),     -- Evidência importa 3x mais que Custo
  ('Evidência', 'Legitimidade', 2),
  ('Legitimidade', 'Custo operacional', 2)       -- o par pode vir em qualquer ordem; 1/3 vale
AS t(criterio_a, criterio_b, valor);

SELECT * FROM gold.lab1_julgamentos

In [0]:
%sql
-- 🟢 PRONTA — pesos AHP (média geométrica) e razão de consistência (CR), em SQL. Só execute.
-- Resultado salvo em gold.lab1_pesos.
CREATE OR REPLACE TABLE gold.lab1_pesos COMMENT 'Pesos AHP dos critérios do grupo e razão de consistência (CR)' AS
WITH c AS (SELECT criterio FROM gold.lab1_criterios),
matriz AS (                       -- matriz par a par completa: diagonal 1, julgamento e seu inverso
  SELECT l.criterio AS linha, k.criterio AS coluna,
         CASE WHEN l.criterio = k.criterio THEN 1.0 ELSE coalesce(j.valor, 1.0 / r.valor) END AS valor
  FROM c l CROSS JOIN c k
  LEFT JOIN gold.lab1_julgamentos j ON j.criterio_a = l.criterio AND j.criterio_b = k.criterio
  LEFT JOIN gold.lab1_julgamentos r ON r.criterio_a = k.criterio AND r.criterio_b = l.criterio
),
geo AS (SELECT linha AS criterio, exp(avg(ln(valor))) AS g FROM matriz GROUP BY linha),
pesos AS (SELECT criterio, g / sum(g) OVER () AS peso FROM geo),
consistencia AS (                 -- lambda máximo e CR = ((lambda - n) / (n - 1)) / índice aleatório de Saaty
  SELECT count(*) AS n, avg(aw / peso) AS lambda_max
  FROM (SELECT m.linha, sum(m.valor * p.peso) AS aw FROM matriz m JOIN pesos p ON p.criterio = m.coluna GROUP BY m.linha) x
  JOIN pesos p ON p.criterio = x.linha
)
SELECT p.criterio, round(p.peso, 3) AS peso,
       round(CASE WHEN n < 3 THEN 0.0
                  ELSE ((lambda_max - n) / (n - 1)) / CASE n WHEN 3 THEN 0.58 WHEN 4 THEN 0.90 ELSE 1.12 END END, 3) AS cr,
       (SELECT count_if(valor IS NULL) FROM matriz) AS julgamentos_faltando
FROM pesos p CROSS JOIN consistencia;

SELECT criterio, concat(CAST(round(peso * 100) AS INT), '%') AS peso, cr,
       CASE WHEN julgamentos_faltando > 0 THEN 'FALTAM julgamentos: confira se todo par de critérios tem um valor'
            WHEN cr < 0.10 THEN 'Julgamentos coerentes: pode usar os pesos'
            ELSE 'CR acima de 0,10: os julgamentos se contradizem. Revejam o par mais discutido e rodem de novo' END AS leitura
FROM gold.lab1_pesos ORDER BY peso DESC

### Atividade 4.3: indicador e limiar de cada critério (10 min)

Para cada critério, digam **como medir** e **a partir de quando o comando age**. O limiar é escrito agora, antes de ver o dado; se fosse escolhido depois, viraria justificativa.

Exemplo para Evidência: indicador = *variação do roubo de rua no trimestre contra o mesmo trimestre do ano anterior, na CISP (ISP)*; limiar = *reforçar se a alta passar de 15% e não for sazonal*.

In [0]:
%sql
-- Atividade 4.3 — um indicador e um limiar para cada critério de gold.lab1_criterios (mesmos nomes).
CREATE OR REPLACE TABLE gold.lab1_indicadores_limiares COMMENT 'Lab 1: indicador e limiar de cada critério, escritos antes de ver os dados' AS
SELECT * FROM VALUES
  ('Evidência',         'TODO', 'TODO'),
  ('Custo operacional', 'TODO', 'TODO'),
  ('Legitimidade',      'TODO', 'TODO')
AS t(criterio, indicador, limiar);

CREATE OR REPLACE TABLE gold.lab1_acompanhamento COMMENT 'Lab 1: o número que mostra se a decisão funcionou' AS
SELECT 'TODO' AS indicador_de_acompanhamento   -- depois da decisão, que número mostra se funcionou?

---

## Parte 5 — Do peso à decisão (11:40–11:50, se der tempo)

Com os pesos, o comando pontua alternativas. Deem nota de **1 a 5** a cada alternativa em cada critério (em custo, 5 = menor custo). O exemplo da aula compara duas CISPs para alocar 2 viaturas extras; troquem pelas alternativas da decisão do grupo.

In [0]:
%sql
-- Parte 5 — alternativas e notas de 1 a 5 em cada critério (exemplo da aula; troque pelas do grupo).
-- Uma linha por alternativa e critério. Em custo, 5 = menor custo.
CREATE OR REPLACE TABLE gold.lab1_alternativas COMMENT 'Lab 1: notas de cada alternativa em cada critério' AS
SELECT * FROM VALUES
  ('CISP X: alta confirmada pelo ISP',                  'Evidência', 4),
  ('CISP X: alta confirmada pelo ISP',                  'Custo operacional', 2),
  ('CISP X: alta confirmada pelo ISP',                  'Legitimidade', 3),
  ('CISP Y: alta menor, vizinha com efetivo sobrando',  'Evidência', 3),
  ('CISP Y: alta menor, vizinha com efetivo sobrando',  'Custo operacional', 4),
  ('CISP Y: alta menor, vizinha com efetivo sobrando',  'Legitimidade', 5)
AS t(alternativa, criterio, nota)

In [0]:
%sql
-- 🟢 PRONTA — pontuação = soma de nota x peso. Só execute. Resultado salvo em gold.lab1_ranking.
CREATE OR REPLACE TABLE gold.lab1_ranking COMMENT 'Lab 1: alternativas pontuadas pelos pesos AHP do grupo' AS
SELECT a.alternativa,
       round(sum(a.nota * p.peso), 2) AS pontuacao,
       concat_ws(' · ', collect_list(concat(a.criterio, ' ', a.nota))) AS notas,
       count(p.criterio) = (SELECT count(*) FROM gold.lab1_criterios) AS todas_as_notas
FROM gold.lab1_alternativas a
LEFT JOIN gold.lab1_pesos p ON p.criterio = a.criterio
GROUP BY a.alternativa;

SELECT rank() OVER (ORDER BY pontuacao DESC) AS posicao, alternativa, pontuacao, notas,
       CASE WHEN todas_as_notas THEN '' ELSE 'FALTA nota em algum critério (confira os nomes)' END AS aviso
FROM gold.lab1_ranking ORDER BY pontuacao DESC

In [0]:
%sql
-- Atividade 5.2 — a decisão preliminar do grupo (opcional hoje; volta na aula 4).
CREATE OR REPLACE TABLE gold.lab1_decisao_preliminar COMMENT 'Lab 1: decisão preliminar, indicador e risco de errar' AS
SELECT
  'TODO' AS alternativa_escolhida,    -- a que o grupo defenderia hoje, mesmo que não seja a do ranking
  'TODO' AS indicador_que_sustenta,   -- ex.: 'variação do roubo de rua na CISP contra o mesmo trimestre do ano anterior'
  'TODO' AS risco_de_errar            -- o que acontece se o dado estiver errado ou a decisão falhar, e como vamos perceber

**Para discutir:** o resultado bate com a intuição do grupo? Se não bate, o desacordo está nos **pesos** ou nas **notas**? Que peso de Evidência faria a outra alternativa vencer?

---

## Parte 6 — Checagem final

Rode as duas células abaixo. A primeira lista o que ainda falta; a segunda monta, a partir das tabelas do grupo, o resumo para levar à aula 2.

In [0]:
%sql
-- 🟢 PRONTA — confere o que falta. Só execute.
-- Se aparecer TABLE_OR_VIEW_NOT_FOUND, a célula daquela parte ainda não foi executada.
SELECT item, CASE WHEN coalesce(ok, false) THEN 'OK' ELSE 'FALTA' END AS situacao
FROM (
  SELECT 1, 'Parte 0 — espaço blue-blood com a série do ISP na Silver', (SELECT count(*) > 0 FROM silver.isp_ocorrencias)
  UNION ALL SELECT 2, 'Parte 0 — número do grupo (1 a 8)', (SELECT max(grupo) BETWEEN 1 AND 8 FROM gold.lab1_grupo)
  UNION ALL SELECT 3, 'Parte 0 — os 5 papéis com nome', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab1_alto_comando)
  UNION ALL SELECT 4, 'Parte 1 — porta escolhida e justificativa', (SELECT max(porta_escolhida IN ('A', 'B', 'C', 'D', 'E') AND justificativa NOT IN ('TODO', '')) FROM gold.lab1_rodada1)
  UNION ALL SELECT 5, 'Parte 1 — pontos revelados', (SELECT max(pontos BETWEEN 0 AND 3) FROM gold.lab1_rodada1_pontos)
  UNION ALL SELECT 6, 'Parte 2 — dado mais crítico', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab1_dado_mais_critico)
  UNION ALL SELECT 7, 'Parte 3 — decisão nomeada', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab1_decisao)
  UNION ALL SELECT 8, 'Parte 4 — julgamentos completos e coerentes (CR < 0,10)', (SELECT max(cr) < 0.10 AND max(julgamentos_faltando) = 0 FROM gold.lab1_pesos)
  UNION ALL SELECT 9, 'Parte 4 — indicador e limiar para cada critério', (SELECT count(i.criterio) = count(*) AND count_if(i.indicador = 'TODO' OR i.limiar = 'TODO') = 0 FROM gold.lab1_criterios c LEFT JOIN gold.lab1_indicadores_limiares i ON i.criterio = c.criterio)
  UNION ALL SELECT 10, 'Parte 4 — indicador de acompanhamento', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab1_acompanhamento)
) AS t(ordem, item, ok)
ORDER BY ordem

In [0]:
%sql
-- 🟢 PRONTA — resumo para a aula 2, lido das tabelas do grupo. Só execute.
SELECT 1 AS ordem, concat('Grupo ', g.grupo, ' — ', d.bairro_ou_cisp, ', ', d.delito) AS resumo
FROM gold.lab1_grupo g CROSS JOIN gold.lab1_decisao d
UNION ALL
SELECT 2, concat('Decisão em jogo: ', decisao_em_jogo, ' (decide: ', quem_decide, '; prazo: ', prazo, ')') FROM gold.lab1_decisao
UNION ALL
SELECT 3, concat('- ', p.criterio, ' (', CAST(round(p.peso * 100) AS INT), '%): ', coalesce(i.indicador, 'TODO'), ' | limiar: ', coalesce(i.limiar, 'TODO'))
FROM gold.lab1_pesos p LEFT JOIN gold.lab1_indicadores_limiares i ON i.criterio = p.criterio
UNION ALL
SELECT 4, concat('Acompanhamento: ', indicador_de_acompanhamento) FROM gold.lab1_acompanhamento
UNION ALL
SELECT 5, CASE WHEN to_json(struct(*)) LIKE '%TODO%'
               THEN 'Decisão preliminar e risco de errar (Parte 5, opcional): ainda em aberto; voltam na aula 4.'
               ELSE concat('Decisão preliminar: ', alternativa_escolhida, ' | indicador: ', indicador_que_sustenta,
                           ' | risco: ', risco_de_errar) END
FROM gold.lab1_decisao_preliminar
UNION ALL
SELECT 6, 'Aviso: os critérios e julgamentos ainda são os do exemplo da aula. Confirmem se o grupo concorda com eles.'
WHERE (SELECT count(*) FROM gold.lab1_julgamentos WHERE criterio_a = 'Evidência' AND criterio_b = 'Custo operacional' AND valor = 3) = 1
  AND (SELECT count(*) FROM gold.lab1_criterios) = 3
ORDER BY ordem

## Material de apoio

- **Slides da aula 1**: o arco da disciplina, critérios mensuráveis, AHP passo a passo, governança e ecossistema de dados de segurança.
- SAATY, T. L. Decision making with the analytic hierarchy process. *International Journal of Services Sciences*, v. 1, n. 1, p. 83–98, 2008 — escala fundamental e consistência.
- DAMA-DMBOK2, cap. 3 (Governança de Dados) — leitura opcional para a aula 2.
- [ISPDados](https://www.ispdados.rj.gov.br/) — séries públicas por CISP, para escolher o recorte do grupo.
- Databricks: [catálogos, schemas e volumes no Unity Catalog](https://docs.databricks.com/aws/en/data-governance/unity-catalog/) e [`read_files`](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files).

**O que fica no espaço `blue-blood`:** na Bronze, os arquivos do ISP e o registro de ingestão; na Silver, a série padronizada e a população por CISP; na Gold, o cardápio de recortes e todas as tabelas `lab1_` com as decisões do grupo. Na aula 2, o lab 2 lê e acrescenta tabelas no mesmo espaço.

**Na aula 2 (05/11, presencial):** arquiteturas de dados, NLP em boletins, julgamentos com probabilidade (System One e `ai_decide` no Databricks), dashboards e normalização por população; Rodada 2 da Operação Manchete.